# 03 - Análisis de nulos y mecanismos de ausencia (copia con tests)

**ORDEN FINAL.** Flujo del notebook: diagnóstico inicial, ajustes puntuales, segunda lectura post-ajustes, tratamiento de nulos y guardado final.


In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 140)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 180)


def find_repo_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError('No se encontro la raiz del repo. Ejecutar dentro del proyecto.')

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
EXTERNAL_DIR = REPO_ROOT / 'Fuentes_de_enriquecimiento'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)


## Carga de datasets preprocesados


In [ ]:
PROCESSED_SOURCES = {
    'df_preprocesado_meli_alq': PROCESSED_DIR / 'df_preprocesado_meli_alq_v1.csv',
    'df_preprocesado_meli_alq_temp': PROCESSED_DIR / 'df_preprocesado_meli_alq_temp_v1.csv',
    'df_preprocesado_meli_vtas': PROCESSED_DIR / 'df_preprocesado_meli_vtas_v1.csv',
    'df_preprocesado_argenprop': PROCESSED_DIR / 'df_preprocesado_argenprop_v1.csv',
    'df_preprocesado_zonaprop': PROCESSED_DIR / 'df_preprocesado_zonaprop_v1.csv',
    'df_preprocesado_airbnb_temp': PROCESSED_DIR / 'df_preprocesado_airbnb_temp_v1.csv',
}

missing = [str(p.relative_to(REPO_ROOT)) for p in PROCESSED_SOURCES.values() if not p.exists()]
if missing:
    raise FileNotFoundError('Faltan archivos procesados. Ejecutar primero 02_normalizacion_y_limpieza.ipynb: ' + ', '.join(missing))

preprocesados = {}
for name, path in PROCESSED_SOURCES.items():
    preprocesados[name] = pd.read_csv(path, low_memory=False)
    globals()[name] = preprocesados[name]
    print(name, preprocesados[name].shape)


## 0. Estrategia a utilizar

Se trabaja en forma progresiva: primero se observan los faltantes, después se aplican ajustes puntuales, luego se vuelve a mirar el efecto de esos ajustes y recién al final se tratan columnas y valores faltantes.


## Funciones para flags y gráficos de faltantes


In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False


def add_null_flags(df, exclude=None):
    out = df.copy()
    exclude = set(exclude or [])
    original_cols = [c for c in out.columns if c not in exclude and not c.endswith('_is_null')]
    for col in original_cols:
        if out[col].isna().any():
            out[f'{col}_is_null'] = out[col].isna().astype('int8')
    return out


def seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=18):
    flag_cols = [c for c in df.columns if c.endswith('_is_null')]
    if not flag_cols:
        return []
    pct = df[flag_cols].mean().mul(100).sort_values(ascending=False)
    return pct[(pct >= min_pct) & (pct <= max_pct)].head(max_flags).index.tolist()


def seleccionar_cuantitativas_informativas(df, max_vars=14):
    candidatas_prioritarias = [
        'precio', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado',
        'superficie_ref_m2', 'superficie_min_m2', 'superficie_max_m2', 'superficie_total_m2',
        'superficie_cubierta_m2', 'ambientes_ref', 'ambientes', 'ambientes_min', 'ambientes_max',
        'dormitorios_ref', 'dormitorios', 'banios_ref', 'banios', 'comuna', 'altura',
        'latitud', 'longitud', 'rating', 'reviews', 'cantidad_imagenes', 'capacidad', 'camas'
    ]
    cols = []
    for col in candidatas_prioritarias:
        if col not in df.columns:
            continue
        if not pd.api.types.is_numeric_dtype(df[col]):
            continue
        if df[col].notna().mean() < 0.05:
            continue
        if df[col].nunique(dropna=True) <= 1:
            continue
        cols.append(col)
    return cols[:max_vars]


def plot_heatmap_matriz(data, title, figsize=(12, 8), annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, mostrar_nan_como_cero=True):
    if data.empty:
        print(f'No hay datos suficientes para graficar: {title}')
        return
    data_plot = data.copy()
    if mostrar_nan_como_cero:
        data_plot = data_plot.fillna(0)
    plt.figure(figsize=figsize)
    if HAS_SEABORN:
        sns.heatmap(data_plot, annot=annot, fmt=fmt, cmap=cmap, center=center, vmin=vmin, vmax=vmax, linewidths=0.5, linecolor='white', cbar_kws={'label': 'Correlacion'})
    else:
        plt.imshow(data_plot, cmap=cmap, vmin=vmin, vmax=vmax)
        plt.colorbar(label='Correlacion')
        plt.xticks(range(len(data_plot.columns)), data_plot.columns, rotation=90)
        plt.yticks(range(len(data_plot.index)), data_plot.index)
    plt.title(title)
    plt.tight_layout()
    plt.show()


def seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10):
    flag_cols = [c for c in df.columns if c.endswith('_is_null')]
    if not flag_cols:
        return []
    resumen = df[flag_cols].mean().mul(100).sort_values(ascending=False).reset_index()
    resumen.columns = ['flag', 'pct_faltante']
    resumen = resumen[(resumen['pct_faltante'] >= min_pct) & (resumen['pct_faltante'] <= max_pct)]
    return resumen.head(top_n)['flag'].tolist()


def seleccionar_categoricas_informativas(df, max_cols=6):
    candidatas_prioritarias = [
        'fuente_norm', 'tipo_operacion_norm', 'moneda_norm', 'barrio_norm', 'tipo_propiedad',
        'localidad', 'provincia', 'inmobiliaria', 'seller_superhost', 'precio_sospechoso',
        'unidad_precio', 'precio_regimen', 'categoria_alojamiento', 'room_type'
    ]
    cols = []
    for col in candidatas_prioritarias:
        if col not in df.columns:
            continue
        n_labels = df[col].nunique(dropna=True)
        pct_no_nulo = df[col].notna().mean()
        if pct_no_nulo < 0.05:
            continue
        if n_labels < 2 or n_labels > 80:
            continue
        cols.append(col)
    return cols[:max_cols]


def faltantes_por_labels_categoricos(df, cat_cols, flag_cols, dataset_name, min_count=30, top_labels_por_col=25):
    rows = []
    for cat_col in cat_cols:
        if cat_col not in df.columns:
            continue
        serie_cat = df[cat_col].astype('string').fillna(f'NULO_EN_{cat_col}')
        labels_validos = serie_cat.value_counts(dropna=False).head(top_labels_por_col).index
        for label in labels_validos:
            mask = serie_cat == label
            n = int(mask.sum())
            if n < min_count:
                continue
            for flag_col in flag_cols:
                if flag_col not in df.columns:
                    continue
                if flag_col == f'{cat_col}_is_null':
                    continue
                rows.append({
                    'dataset': dataset_name,
                    'variable_categorica': cat_col,
                    'label': label,
                    'n_registros': n,
                    'variable_faltante': flag_col,
                    'pct_faltante': round(df.loc[mask, flag_col].mean() * 100, 2),
                })
    return pd.DataFrame(rows)


def graficar_faltantes_por_label(df_resumen, dataset_name):
    if df_resumen.empty:
        print(f'No se genero tabla de faltantes por label para {dataset_name}.')
        return
    df_pivot = df_resumen.pivot_table(
        index=['variable_categorica', 'label'],
        columns='variable_faltante',
        values='pct_faltante',
        aggfunc='mean'
    ).fillna(0)
    if df_pivot.empty:
        print(f'No hay datos suficientes para graficar {dataset_name}.')
        return
    variables_unicas = list(df_pivot.index.get_level_values('variable_categorica').unique())
    alto = min(22, max(7, len(df_pivot) * 0.32))
    if HAS_SEABORN:
        paleta = sns.color_palette('Set2', n_colors=len(variables_unicas))
        mapa_colores = dict(zip(variables_unicas, paleta))
        colores_filas = [mapa_colores[v] for v in df_pivot.index.get_level_values('variable_categorica')]
        g = sns.clustermap(
            df_pivot,
            row_cluster=False,
            col_cluster=False,
            row_colors=colores_filas,
            cmap='YlOrRd',
            annot=True,
            fmt='.1f',
            linewidths=0.5,
            figsize=(14, alto),
        )
        g.fig.suptitle(f'Porcentaje de faltantes por label categorico - {dataset_name}', y=1.02)
        plt.show()
    else:
        plt.figure(figsize=(14, alto))
        plt.imshow(df_pivot.values, aspect='auto')
        plt.colorbar(label='% faltante')
        plt.xticks(range(len(df_pivot.columns)), df_pivot.columns, rotation=90)
        plt.yticks(range(len(df_pivot.index)), [f'{a} = {b}' for a, b in df_pivot.index], fontsize=8)
        plt.title(f'Porcentaje de faltantes por label categorico - {dataset_name}')
        plt.tight_layout()
        plt.show()


## Creación de flags de nulidad por dataset original


In [ ]:
preprocesados_con_flags = {}
for name, df in preprocesados.items():
    preprocesados_con_flags[name] = add_null_flags(df)
    globals()[name] = preprocesados_con_flags[name]
    print(name, 'flags creados:', sum(c.endswith('_is_null') for c in preprocesados_con_flags[name].columns))


## Celdas de decisión: estrategia inicial de nulos

Antes de aplicar ajustes, se conserva una estrategia conservadora: flaggear y analizar. La imputación o eliminación se decide después de revisar los patrones.


In [ ]:
# Estrategia conservadora por defecto.
# Opciones esperadas: 'flaggear', 'eliminar_criticos'
ESTRATEGIA_NULOS = 'flaggear'
CAMPOS_CRITICOS_NULOS = ['precio', 'moneda_norm', 'barrio_norm', 'tipo_operacion_norm']

resumen_estrategia_nulos = []
for name, df in preprocesados_con_flags.items():
    campos_existentes = [c for c in CAMPOS_CRITICOS_NULOS if c in df.columns]
    filas_antes = len(df)
    filas_con_nulo_critico = int(df[campos_existentes].isna().any(axis=1).sum()) if campos_existentes else 0

    if ESTRATEGIA_NULOS == 'eliminar_criticos' and campos_existentes:
        preprocesados_con_flags[name] = df.dropna(subset=campos_existentes).copy()
        accion = 'eliminados registros con nulos en campos criticos'
    else:
        accion = 'solo flags; no se eliminan registros'

    resumen_estrategia_nulos.append({
        'dataset': name,
        'estrategia': ESTRATEGIA_NULOS,
        'campos_criticos_existentes': ', '.join(campos_existentes),
        'filas_antes': filas_antes,
        'filas_con_nulo_critico': filas_con_nulo_critico,
        'filas_despues': len(preprocesados_con_flags[name]),
        'accion': accion,
    })

resumen_estrategia_nulos = pd.DataFrame(resumen_estrategia_nulos)
print('Reporte de estrategia de nulos guardado:', (REPORTS_DIR / 'decisiones_tratamiento_nulos_v1.csv').relative_to(REPO_ROOT))
resumen_estrategia_nulos.to_csv(REPORTS_DIR / 'decisiones_tratamiento_nulos_v1.csv', index=False)


## 1. Diagnóstico inicial: matrices de faltantes por dataset

Primero se observan los dataframes tal como llegan desde la normalización inicial. En esta etapa todavía no se corrigen columnas ni se separan Argenprop/Zonaprop por operación.


In [ ]:
for name, df in preprocesados_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags = seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=16)
    cuantitativas = seleccionar_cuantitativas_informativas(df, max_vars=12)

    if len(flags) < 2:
        print('No hay suficientes flags informativos de faltantes para este dataset.')
        continue

    matriz_faltante_faltante = df[flags].corr()
    matriz_faltante_faltante.to_csv(REPORTS_DIR / f'matriz_faltante_vs_faltante_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_faltante, title=f'Faltante vs faltante - {name}', figsize=(12, 9), annot=True, mostrar_nan_como_cero=True)

    if len(cuantitativas) == 0:
        print('No hay variables cuantitativas informativas para cruzar contra faltantes.')
        continue

    matriz_faltante_cuantitativa = df[flags + cuantitativas].corr(numeric_only=True).loc[flags, cuantitativas]
    umbral_corr_cuantitativas = 0.15
    matriz_faltante_cuantitativa_filtrada = matriz_faltante_cuantitativa.where(matriz_faltante_cuantitativa.abs() >= umbral_corr_cuantitativas, other=0)
    matriz_faltante_cuantitativa.to_csv(REPORTS_DIR / f'matriz_faltante_vs_cuantitativas_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_cuantitativa_filtrada, title=f'Faltante vs variables cuantitativas - {name}', figsize=(13, 8), annot=True, mostrar_nan_como_cero=True)


### 1.1 Validación estadística de asociaciones principales

Las matrices permiten detectar relaciones fuertes entre faltantes, pero no distinguen por sí solas si la asociación observada es estadísticamente consistente. Para no sobrecargar el análisis, se testean solo las relaciones más fuertes:

| Relación | Prueba utilizada | Motivo |
|---|---|---|
| Faltante vs faltante | Chi-cuadrado o Fisher | Ambas variables son binarias. Fisher se usa si hay frecuencias esperadas bajas. |
| Faltante vs variable cuantitativa | Mann-Whitney | Compara la distribución de la variable cuantitativa entre registros con y sin faltante, sin asumir normalidad. |

Estos tests complementan el diagnóstico de calidad del dato. No reemplazan el análisis visual ni prueban hipótesis de negocio; sirven para priorizar qué patrones de ausencia conviene mirar con más atención.


In [ ]:
try:
    from scipy.stats import chi2_contingency, fisher_exact, mannwhitneyu
    SCIPY_DISPONIBLE = True
except Exception:
    SCIPY_DISPONIBLE = False


def tests_faltante_vs_faltante(df, flags, umbral_corr=0.35, max_tests=8):
    matriz = df[flags].corr()
    pares = []

    for i, flag_1 in enumerate(flags):
        for flag_2 in flags[i + 1:]:
            corr = matriz.loc[flag_1, flag_2]
            if pd.notna(corr) and abs(corr) >= umbral_corr:
                pares.append((flag_1, flag_2, corr))

    pares = sorted(pares, key=lambda x: abs(x[2]), reverse=True)[:max_tests]
    resultados = []

    for flag_1, flag_2, corr in pares:
        tabla = pd.crosstab(df[flag_1], df[flag_2])
        if tabla.shape != (2, 2):
            continue

        chi2, p_chi, _, expected = chi2_contingency(tabla)
        if (expected < 5).any():
            _, p_value = fisher_exact(tabla)
            test = 'Fisher'
        else:
            p_value = p_chi
            test = 'Chi-cuadrado'

        resultados.append({
            'variable_1': flag_1,
            'variable_2': flag_2,
            'correlacion_phi': round(corr, 3),
            'test': test,
            'p_value': round(p_value, 5),
            'asociacion_significativa': p_value < 0.05,
        })

    return pd.DataFrame(resultados)


def tests_faltante_vs_cuantitativa(df, flags, cuantitativas, umbral_corr=0.15, max_tests=8):
    matriz = df[flags + cuantitativas].corr(numeric_only=True).loc[flags, cuantitativas]
    pares = []

    for flag in flags:
        for col in cuantitativas:
            corr = matriz.loc[flag, col]
            if pd.notna(corr) and abs(corr) >= umbral_corr:
                pares.append((flag, col, corr))

    pares = sorted(pares, key=lambda x: abs(x[2]), reverse=True)[:max_tests]
    resultados = []

    for flag, col, corr in pares:
        grupo_faltante = df.loc[df[flag] == 1, col].dropna()
        grupo_no_faltante = df.loc[df[flag] == 0, col].dropna()

        if len(grupo_faltante) < 20 or len(grupo_no_faltante) < 20:
            continue

        stat, p_value = mannwhitneyu(grupo_faltante, grupo_no_faltante, alternative='two-sided')

        resultados.append({
            'flag_faltante': flag,
            'variable_cuantitativa': col,
            'correlacion': round(corr, 3),
            'mediana_con_faltante': round(grupo_faltante.median(), 2),
            'mediana_sin_faltante': round(grupo_no_faltante.median(), 2),
            'diferencia_mediana': round(grupo_faltante.median() - grupo_no_faltante.median(), 2),
            'test': 'Mann-Whitney',
            'p_value': round(p_value, 5),
            'diferencia_significativa': p_value < 0.05,
        })

    return pd.DataFrame(resultados)


resultados_tests_faltantes = []

if SCIPY_DISPONIBLE:
    for name, df in preprocesados_con_flags.items():
        flags = seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=16)
        cuantitativas = seleccionar_cuantitativas_informativas(df, max_vars=12)

        if len(flags) >= 2:
            df_test_ff = tests_faltante_vs_faltante(df, flags)
            if not df_test_ff.empty:
                df_test_ff.insert(0, 'dataset', name)
                df_test_ff.insert(1, 'tipo_relacion', 'faltante_vs_faltante')
                resultados_tests_faltantes.append(df_test_ff)

        if len(flags) >= 1 and len(cuantitativas) >= 1:
            df_test_fc = tests_faltante_vs_cuantitativa(df, flags, cuantitativas)
            if not df_test_fc.empty:
                df_test_fc.insert(0, 'dataset', name)
                df_test_fc.insert(1, 'tipo_relacion', 'faltante_vs_cuantitativa')
                resultados_tests_faltantes.append(df_test_fc)

    if resultados_tests_faltantes:
        df_tests_faltantes = pd.concat(resultados_tests_faltantes, ignore_index=True)
        display(df_tests_faltantes)
        df_tests_faltantes.to_csv(REPORTS_DIR / 'tests_asociacion_faltantes_iniciales.csv', index=False)
    else:
        print('No se encontraron relaciones suficientemente fuertes para testear.')
else:
    print('scipy no esta disponible. Se omiten los tests estadisticos y se conserva la lectura visual de matrices.')


### 1.1 Conclusiones de correlaciones originales

En esta primera lectura aparecen mezcladas publicaciones de venta y alquiler dentro de una misma fuente. Eso puede inflar o esconder relaciones entre faltantes, porque una operación puede tener campos naturalmente más presentes que otra. Por eso se decide generar una segunda lectura separando por `tipo_operacion_norm`.

#### MeLi alquiler
Se observan faltantes vinculados a campos poco cargados por el portal, como rangos de dormitorios o información de entrega/posesión.

#### Argenprop y Zonaprop combinados
La lectura combinada no es suficiente porque mezcla venta y alquiler. Se decide separar ambos datasets por operación antes de tomar decisiones definitivas.

#### Las Cañitas
Aparece `Las Cañitas / las canitas` como barrio operativo con comuna faltante. Se corrige después de observarlo: `barrio_norm = palermo` y `comuna = 14`.

#### Localidad
En algunos dataframes `localidad` toma siempre el mismo valor asociado a CABA. Como el alcance del TP ya está definido sobre propiedades en CABA, esa columna no agrega segmentación.

#### Moneda
Aunque en algunos dataframes `moneda_norm` tenga una sola etiqueta, se conserva porque en otros datasets sí varía y es necesaria para comparar precios.


### 1.2 Faltantes por label de variables categóricas

Además de las matrices, se revisan faltantes contra labels categóricos dentro de cada dataset. Esto permite detectar problemas puntuales por barrio, tipo de operación, fuente u otra categoría.


In [ ]:
resumenes_faltantes_por_label = []
for name, df in preprocesados_con_flags.items():
    print('\n' + '=' * 100)
    print(name)

    flags_diagnostico = seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10)
    cat_cols_diagnostico = seleccionar_categoricas_informativas(df, max_cols=6)

    if not flags_diagnostico:
        print('No hay flags de faltantes informativos para este dataset.')
        continue
    if not cat_cols_diagnostico:
        print('No hay variables categoricas informativas para este dataset.')
        continue

    df_faltantes_label_dataset = faltantes_por_labels_categoricos(df, cat_cols_diagnostico, flags_diagnostico, dataset_name=name, min_count=30, top_labels_por_col=25)
    if df_faltantes_label_dataset.empty:
        print('No hay combinaciones con volumen suficiente para graficar.')
        continue

    resumenes_faltantes_por_label.append(df_faltantes_label_dataset)
    df_faltantes_label_dataset.to_csv(REPORTS_DIR / f'faltantes_por_label_categorico_{name}.csv', index=False)
    graficar_faltantes_por_label(df_faltantes_label_dataset, dataset_name=name)

if resumenes_faltantes_por_label:
    df_faltantes_por_label_todos = pd.concat(resumenes_faltantes_por_label, ignore_index=True)
    df_faltantes_por_label_todos.to_csv(REPORTS_DIR / 'faltantes_por_label_categorico_todos_los_datasets.csv', index=False)
    print('Reporte consolidado guardado:', (REPORTS_DIR / 'faltantes_por_label_categorico_todos_los_datasets.csv').relative_to(REPO_ROOT))


### 1.3 Conclusiones de faltantes por labels originales

A partir de las matrices y gráficos iniciales se decide:

1. Separar Argenprop y Zonaprop por tipo de operación.
2. Corregir puntualmente `Las Cañitas / las canitas` como Palermo y completar comuna 14.
3. Eliminar `localidad` cuando sea constante CABA.
4. Conservar `moneda_norm`, aunque en algunos dataframes tenga una sola etiqueta.


In [ ]:
# Guardado del diagnostico original con flags.
# Estos archivos conservan el estado previo a los ajustes decididos a partir de las matrices.
for name, df in preprocesados_con_flags.items():
    path = PROCESSED_DIR / f'{name}_con_flags_nulos_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado diagnostico original:', path.relative_to(REPO_ROOT), df.shape)


## 2. Ajustes derivados del diagnóstico inicial

Estos ajustes aparecen después del diagnóstico porque surgen de lo observado en matrices y tablas por label. No se oculta el problema original: primero se lo muestra y después se corrige.


In [ ]:
# Se parte de copias para conservar el diagnostico original sin pisarlo.
preprocesados_post_diagnostico = {name: df.copy() for name, df in preprocesados.items()}


### 2.1 Corrección puntual de Las Cañitas

El diagnóstico mostró que `Las Cañitas / las canitas` aparece como barrio operativo con `comuna` faltante. La corrección es puntual y simple: reemplazarlo por `palermo` y asignar `comuna = 14`. No se aplica ningún mapeo general de barrios en esta etapa.


In [ ]:
# Correccion simple: Las Cañitas pertenece operativamente a Palermo, comuna 14.
for name, df in preprocesados_post_diagnostico.items():
    if 'barrio_norm' in df.columns and 'comuna' in df.columns:
        mask_canitas = df['barrio_norm'].astype('string').str.lower().str.strip().isin(['las cañitas', 'las canitas'])
        df.loc[mask_canitas, 'barrio_norm'] = 'palermo'
        df.loc[mask_canitas, 'comuna'] = 14


### 2.2 Eliminación de `localidad` constante

Como el TP trabaja sobre propiedades en CABA, `localidad` se elimina solo cuando tiene una única etiqueta equivalente a CABA. Si la columna tuviera más de una localidad, no se elimina en esta regla.


In [ ]:
valores_caba = {'caba', 'capital federal', 'ciudad autonoma de buenos aires', 'ciudad autónoma de buenos aires'}

for name, df in list(preprocesados_post_diagnostico.items()):
    if 'localidad' in df.columns:
        localidades = set(df['localidad'].dropna().astype('string').str.lower().str.strip().unique())
        if len(localidades) <= 1 and localidades.issubset(valores_caba):
            preprocesados_post_diagnostico[name] = df.drop(columns=['localidad'])
            print(name, 'localidad eliminada por ser constante CABA')


### 2.3 Separación de Argenprop y Zonaprop por operación

Después de los ajustes puntuales, Argenprop y Zonaprop se separan en venta y alquiler para que los faltantes de una operación no contaminen la lectura de la otra.


In [ ]:
def separar_por_operacion(nombre_base, df, mapeo_operaciones):
    resultado = {}
    categorias_presentes = set(df['tipo_operacion_norm'].dropna().unique())
    categorias_sin_mapear = categorias_presentes - set(mapeo_operaciones)
    if categorias_sin_mapear:
        print(f'AVISO [{nombre_base}]: categorias sin mapear, se excluyen: {categorias_sin_mapear}')
    for valor_operacion, sufijo in mapeo_operaciones.items():
        sub_df = df[df['tipo_operacion_norm'] == valor_operacion].copy()
        clave = f'{nombre_base}_{sufijo}'
        resultado[clave] = sub_df
        print(clave, sub_df.shape)
    return resultado

MAPEO_OPERACIONES_PORTALES = {'venta': 'vtas', 'alquiler': 'alq'}

preprocesados_post_ajustes = {
    name: df for name, df in preprocesados_post_diagnostico.items()
    if name not in {'df_preprocesado_argenprop', 'df_preprocesado_zonaprop'}
}
preprocesados_post_ajustes.update(separar_por_operacion('df_preprocesado_argenprop', preprocesados_post_diagnostico['df_preprocesado_argenprop'], MAPEO_OPERACIONES_PORTALES))
preprocesados_post_ajustes.update(separar_por_operacion('df_preprocesado_zonaprop', preprocesados_post_diagnostico['df_preprocesado_zonaprop'], MAPEO_OPERACIONES_PORTALES))

preprocesados_post_ajustes_con_flags = {}
for name, df in preprocesados_post_ajustes.items():
    preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
    globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]

print('\nDatasets generados post-ajustes:')
for name, df in preprocesados_post_ajustes_con_flags.items():
    print(name, df.shape)


## 3. Segunda lectura post-ajustes

Después de corregir `Las Cañitas`, eliminar `localidad` constante y separar Argenprop/Zonaprop por operación, se vuelven a mirar únicamente las matrices y gráficos necesarios para evaluar esos cambios.


In [ ]:
DATASETS_POST_AJUSTES_A_COMPARAR = [
    'df_preprocesado_argenprop_vtas',
    'df_preprocesado_argenprop_alq',
    'df_preprocesado_zonaprop_vtas',
    'df_preprocesado_zonaprop_alq',
]

preprocesados_post_ajustes_para_comparar = {
    name: preprocesados_post_ajustes_con_flags[name]
    for name in DATASETS_POST_AJUSTES_A_COMPARAR
    if name in preprocesados_post_ajustes_con_flags
}

print('Datasets post-ajustes incluidos en la comparacion:')
for name, df in preprocesados_post_ajustes_para_comparar.items():
    print(name, df.shape)


### 3.1 Matrices post-ajustes para comparar

Estas matrices son una segunda lectura. No reemplazan las matrices originales: sirven para verificar si los ajustes reducen faltantes artificiales o separan patrones que antes estaban mezclados.


In [ ]:
for name, df in preprocesados_post_ajustes_para_comparar.items():
    print('\n' + '=' * 100)
    print(name)

    flags = seleccionar_flags_informativos(df, min_pct=2, max_pct=98, max_flags=16)
    cuantitativas = seleccionar_cuantitativas_informativas(df, max_vars=12)

    if len(flags) < 2:
        print('No hay suficientes flags informativos de faltantes para este dataset.')
        continue

    matriz_faltante_faltante = df[flags].corr()
    matriz_faltante_faltante.to_csv(REPORTS_DIR / f'matriz_faltante_vs_faltante_post_ajustes_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_faltante, title=f'POST AJUSTES - Faltante vs faltante - {name}', figsize=(12, 9), annot=True, mostrar_nan_como_cero=True)

    if len(cuantitativas) == 0:
        print('No hay variables cuantitativas informativas para cruzar contra faltantes.')
        continue

    matriz_faltante_cuantitativa = df[flags + cuantitativas].corr(numeric_only=True).loc[flags, cuantitativas]
    umbral_corr_cuantitativas = 0.15
    matriz_faltante_cuantitativa_filtrada = matriz_faltante_cuantitativa.where(matriz_faltante_cuantitativa.abs() >= umbral_corr_cuantitativas, other=0)
    matriz_faltante_cuantitativa.to_csv(REPORTS_DIR / f'matriz_faltante_vs_cuantitativas_post_ajustes_{name}.csv')
    plot_heatmap_matriz(matriz_faltante_cuantitativa_filtrada, title=f'POST AJUSTES - Faltante vs variables cuantitativas - {name}', figsize=(13, 8), annot=True, mostrar_nan_como_cero=True)


### 3.2 Faltantes por label post-ajustes

Se repite el cruce contra variables categóricas solo para los datasets nuevos separados. El objetivo no es volver a perfilar todo, sino revisar el efecto de separar Argenprop y Zonaprop por operación.


In [ ]:
resumenes_faltantes_por_label_post = []
for name, df in preprocesados_post_ajustes_para_comparar.items():
    print('\n' + '=' * 100)
    print(name)

    flags_diagnostico = seleccionar_flags_problematicos(df, min_pct=2, max_pct=98, top_n=10)
    cat_cols_diagnostico = seleccionar_categoricas_informativas(df, max_cols=6)

    if not flags_diagnostico:
        print('No hay flags de faltantes informativos para este dataset.')
        continue
    if not cat_cols_diagnostico:
        print('No hay variables categoricas informativas para este dataset.')
        continue

    df_faltantes_label_dataset = faltantes_por_labels_categoricos(df, cat_cols_diagnostico, flags_diagnostico, dataset_name=name, min_count=30, top_labels_por_col=25)
    if df_faltantes_label_dataset.empty:
        print('No hay combinaciones con volumen suficiente para graficar.')
        continue

    resumenes_faltantes_por_label_post.append(df_faltantes_label_dataset)
    df_faltantes_label_dataset.to_csv(REPORTS_DIR / f'faltantes_por_label_categorico_post_ajustes_{name}.csv', index=False)
    graficar_faltantes_por_label(df_faltantes_label_dataset, dataset_name=f'POST AJUSTES - {name}')

if resumenes_faltantes_por_label_post:
    df_faltantes_por_label_post_todos = pd.concat(resumenes_faltantes_por_label_post, ignore_index=True)
    df_faltantes_por_label_post_todos.to_csv(REPORTS_DIR / 'faltantes_por_label_categorico_post_ajustes_todos_los_datasets.csv', index=False)
    print('Reporte post-ajustes guardado:', (REPORTS_DIR / 'faltantes_por_label_categorico_post_ajustes_todos_los_datasets.csv').relative_to(REPO_ROOT))


### 3.3 Cierre de la segunda lectura

La segunda lectura confirma el efecto de los ajustes: Argenprop y Zonaprop ya se analizan separados por operación, y el caso de `Las Cañitas` queda corregido como Palermo/comuna 14.

A partir de este punto ya no se busca volver a comparar matrices, sino pasar al tratamiento de columnas y valores faltantes.


## 4. Tratamiento de nulos

A esta altura ya entendemos mejor los faltantes: vimos el diagnóstico original, hicimos los ajustes necesarios y revisamos las matrices post-ajuste. Desde este punto se trabaja con los dataframes ya ajustados: MeLi y Airbnb se mantienen como estaban, y Argenprop/Zonaprop quedan separados en venta y alquiler.

El orden de tratamiento es:

1. eliminar columnas estructurales;
2. eliminar columnas que no sirven para el análisis;
3. imputar solo donde tenga sentido;
4. mantener como susceptibles de reconstrucción las variables recuperables desde texto, atributos u otras columnas.


### 4.1 Eliminación de columnas estructurales

Primero se eliminan columnas con 100% de nulos. En este caso no hay información disponible para trabajar dentro de ese dataframe.

| Caso | Decisión | Motivo |
|---|---|---|
| Columna con 100% de nulos | Eliminar | No tiene datos observados en esa fuente. |
| Columna con muchos nulos pero presente con datos en otros dataframes | Mantener por ahora | Puede servir cuando el dato existe en otras fuentes. |
| Columna con muchos nulos pero recuperable desde texto, atributos u otra columna | Mantener para reconstrucción | Antes de eliminarla conviene intentar recuperarla. |
| Columna con muchos nulos, sin datos en otros dataframes y sin reconstrucción clara | Eliminar | No aporta información usable para esta etapa. |


In [ ]:
# Eliminacion de columnas 100% nulas, ya con Argenprop/Zonaprop separados.

COLUMNAS_A_DROPPEAR_POR_DATASET = {}

for name, df in preprocesados_post_ajustes.items():
    columnas_100_nulas = [
        col for col in df.columns
        if df[col].isna().mean() == 1
    ]

    columnas_a_droppear = sorted(set(columnas_100_nulas))
    COLUMNAS_A_DROPPEAR_POR_DATASET[name] = columnas_a_droppear

    print('\n' + '=' * 100)
    print(name)
    print('Columnas 100% nulas:', columnas_100_nulas if columnas_100_nulas else 'ninguna')
    print('Columnas a droppear ahora:', columnas_a_droppear if columnas_a_droppear else 'ninguna')

APLICAR_DROP_COLUMNAS_ESTRUCTURALES = True

if APLICAR_DROP_COLUMNAS_ESTRUCTURALES:
    preprocesados_post_ajustes = {
        name: df.drop(columns=COLUMNAS_A_DROPPEAR_POR_DATASET.get(name, []), errors='ignore')
        for name, df in preprocesados_post_ajustes.items()
    }
    print('\nDrop de columnas estructurales aplicado.')
else:
    print('\nNo se aplico el drop. Revisar COLUMNAS_A_DROPPEAR_POR_DATASET antes de ejecutar.')

preprocesados_post_ajustes_con_flags = {}
for name, df in preprocesados_post_ajustes.items():
    preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
    globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]


### 4.2 Revisión de columnas con muchos nulos

Después de eliminar las columnas 100% vacías, revisamos las columnas con muchos nulos. El umbral del 80% se usa como alerta, no como regla automática de borrado.

Para cada columna se decide una de estas opciones:

| Decisión | Cuándo aplica |
|---|---|
| `borrar` | Tiene muchos nulos, no aparece con datos en otros dataframes y no hay forma clara de reconstruirla. |
| `mantener_por_otros_dfs` | Tiene muchos nulos en este dataframe, pero aparece con datos en otros dataframes. |
| `reconstruir` | Puede recuperarse desde texto, atributos, URL u otra columna. |


In [ ]:
UMBRAL_NULOS_ALTO = 0.80 # filtros para decidir que columnas se borran
MIN_PCT_DATOS_EN_OTROS_DFS = 0.20 # buscando preservar las columnas mas utiles

# Ajustes manuales globales.
# Si una columna aparece aca, la decision manual pisa la regla automatica.
COLUMNAS_BORRAR_SIEMPRE = {
    # Columnas que se ausentan en bloque o no aportan al alcance actual del trabajo.
    'posesion',
    'estado_entrega',
    'unidades_disponibles',
    'precio_anterior',
    'descuento_pct',
}

# Columnas analiticas que se conservan para imputar o completar con reglas simples.
# La idea es trabajar luego con variables de referencia, no con multiples versiones redundantes.
COLUMNAS_IMPUTAR_SIEMPRE = {
    'cocheras',
    'ambientes_ref',
    'banios_ref',
    'dormitorios_ref',
    'expensas_moneda',
}

# Columnas auxiliares: se usan para completar variables *_ref cuando sirven,
# pero despues se eliminan para evitar duplicar informacion.
COLUMNAS_USAR_PARA_REF_Y_BORRAR = {
    'ambientes',
    'ambientes_min',
    'ambientes_max',
    'banios_min',
    'banios_max',
    'dormitorios_min',
    'dormitorios_max',
}

COLUMNAS_RECONSTRUIR_SIEMPRE = {
    'expensas', # se debe reconstruir a partir del texto o la publicacion, o imputarla de alguna forma mas compleja
    'expensas_texto',
    'superficie_ref_m2',
    'precio_m2_ref',
    'inmobiliaria',
}

COLUMNAS_MANTENER_SIEMPRE = {
    # otras columnas que se quieran mantener sin ideas claras por el momento
}

registros_revision_nulos_altos = []

for name, df in preprocesados_post_ajustes.items():
    otros_dfs = {
        other_name: other_df
        for other_name, other_df in preprocesados_post_ajustes.items()
        if other_name != name
    }

    for col in df.columns:
        pct_nulos = df[col].isna().mean()
        if pct_nulos < UMBRAL_NULOS_ALTO:
            continue

        n_no_nulos = int(df[col].notna().sum())

        aparece_con_datos_en_otros = any(
            col in other_df.columns
            and other_df[col].notna().mean() >= MIN_PCT_DATOS_EN_OTROS_DFS
            for other_df in otros_dfs.values()
        )

        if col in COLUMNAS_BORRAR_SIEMPRE:
            decision_sugerida = 'borrar'
            criterio_decision = 'borrar por decision manual global'

        elif col in COLUMNAS_USAR_PARA_REF_Y_BORRAR:
            decision_sugerida = 'usar_para_ref_y_borrar'
            criterio_decision = 'auxiliar para completar variable de referencia'

        elif col in COLUMNAS_IMPUTAR_SIEMPRE:
            decision_sugerida = 'imputar_ref'
            criterio_decision = 'variable de referencia imputable con regla simple'

        elif col in COLUMNAS_RECONSTRUIR_SIEMPRE:
            decision_sugerida = 'reconstruir'
            criterio_decision = 'susceptible de reconstruccion'

        elif col in COLUMNAS_MANTENER_SIEMPRE:
            decision_sugerida = 'mantener_por_otros_dfs'
            criterio_decision = 'mantener por decision manual global'

        elif aparece_con_datos_en_otros:
            decision_sugerida = 'mantener_por_otros_dfs'
            criterio_decision = 'aparece con datos suficientes en otros dfs'

        else:
            decision_sugerida = 'borrar'
            criterio_decision = 'muchos nulos y sin regla clara de uso'

        registros_revision_nulos_altos.append({
            'dataset': name,
            'columna': col,
            'pct_nulos': round(pct_nulos * 100, 2),
            'n_no_nulos': n_no_nulos,
            'aparece_con_datos_en_otros_dfs': aparece_con_datos_en_otros,
            'decision_sugerida': decision_sugerida,
            'criterio_decision': criterio_decision,
        })

df_revision_nulos_altos = (
    pd.DataFrame(registros_revision_nulos_altos)
    .sort_values(
        ['dataset', 'decision_sugerida', 'pct_nulos', 'columna'],
        ascending=[True, True, False, True]
    )
)

display(df_revision_nulos_altos)


Las columnas con muchos nulos no se tratan todas igual. Si aparecen con datos en otros dataframes, se mantienen. Si pueden reconstruirse, se reservan para una regla posterior. Si no cumplen ninguna de esas condiciones, se eliminan.


In [ ]:
# Aplicacion de decisiones sobre columnas con muchos nulos.
# Se borran solo las marcadas como borrar en la tabla anterior.

COLUMNAS_NULOS_ALTOS_A_DROPPEAR = {}

if not df_revision_nulos_altos.empty:
    df_a_borrar = df_revision_nulos_altos[df_revision_nulos_altos['decision_sugerida'] == 'borrar']
    COLUMNAS_NULOS_ALTOS_A_DROPPEAR = (
        df_a_borrar
        .groupby('dataset')['columna']
        .apply(list)
        .to_dict()
    )

print('Columnas con nulos altos a droppear:')
for name, cols in COLUMNAS_NULOS_ALTOS_A_DROPPEAR.items():
    print(name, cols)

APLICAR_DROP_NULOS_ALTOS = True

if APLICAR_DROP_NULOS_ALTOS:
    for name, cols in COLUMNAS_NULOS_ALTOS_A_DROPPEAR.items():
        if name not in preprocesados_post_ajustes:
            continue
        cols_existentes = [col for col in cols if col in preprocesados_post_ajustes[name].columns]
        preprocesados_post_ajustes[name] = preprocesados_post_ajustes[name].drop(columns=cols_existentes)
        print(name, 'columnas eliminadas por nulos altos:', cols_existentes)

    preprocesados_post_ajustes_con_flags = {}
    for name, df in preprocesados_post_ajustes.items():
        preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
        globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]
else:
    print('No se eliminan columnas con nulos altos automaticamente.')


### 4.3 Resultado del bloque de eliminación

A partir de este punto, los dataframes de trabajo son los contenidos en `preprocesados_post_ajustes` y `preprocesados_post_ajustes_con_flags`. Las columnas eliminadas corresponden a:

- columnas con 100% de nulos;
- columnas con muchos nulos que no aparecen con datos en otros dataframes y no fueron marcadas como reconstruibles o relevantes para el analisis. 


### 4.4 Imputación de datos faltantes donde tenga sentido

Se imputan solo variables que tienen una regla simple y explícita. Para mantener consistencia, se priorizan las variables de referencia (`*_ref`) como columnas analíticas finales. Las columnas auxiliares (`*_min`, `*_max` o versiones originales redundantes) se usan para completar esos valores si aportan información, y luego se eliminan.

| Tipo de variable | Criterio aplicado |
|---|---|
| `cocheras` | Se imputan nulos por `0`, dejando flag de imputación. Si más adelante el texto identifica cochera, puede corregirse. |
| `expensas_moneda` | Si existe `expensas` y falta la moneda, se imputa `ARS`. |
| Variables `*_ref` | Se completan desde una fuente directa: valor original, rango con valor único o regla simple. |
| Rangos `*_min/max` | Se usan como insumo si ayudan a completar `*_ref`, pero no quedan como columnas finales si duplican información. |

No se imputan variables centrales sin evidencia suficiente. En esos casos se mantienen para reconstrucción o se eliminan según la tabla de decisión anterior.


In [ ]:
# Imputaciones controladas sobre columnas seleccionadas.
# No se pisan valores existentes: solo se completan nulos cuando hay una regla simple.

APLICAR_IMPUTACIONES_CONTROLADAS = True

resumen_imputaciones = []


def completar_ref_desde_rango(df, col_ref, col_min, col_max, dataset_name):
    """Completa col_ref cuando el rango permite un valor unico."""
    if col_ref not in df.columns:
        return df

    if col_min in df.columns and col_max in df.columns:
        mask_igual = df[col_ref].isna() & df[col_min].notna() & df[col_max].notna() & (df[col_min] == df[col_max])
        df.loc[mask_igual, col_ref] = df.loc[mask_igual, col_min]
        resumen_imputaciones.append({
            'dataset': dataset_name,
            'columna': col_ref,
            'criterio': f'{col_ref} imputado cuando {col_min} = {col_max}',
            'n_imputados': int(mask_igual.sum())
        })

    if col_min in df.columns:
        mask_desde_min = df[col_ref].isna() & df[col_min].notna()
        df.loc[mask_desde_min, col_ref] = df.loc[mask_desde_min, col_min]
        resumen_imputaciones.append({
            'dataset': dataset_name,
            'columna': col_ref,
            'criterio': f'{col_ref} imputado desde {col_min} cuando no hay rango completo',
            'n_imputados': int(mask_desde_min.sum())
        })

    if col_max in df.columns:
        mask_desde_max = df[col_ref].isna() & df[col_max].notna()
        df.loc[mask_desde_max, col_ref] = df.loc[mask_desde_max, col_max]
        resumen_imputaciones.append({
            'dataset': dataset_name,
            'columna': col_ref,
            'criterio': f'{col_ref} imputado desde {col_max} cuando no hay rango completo',
            'n_imputados': int(mask_desde_max.sum())
        })

    return df


if APLICAR_IMPUTACIONES_CONTROLADAS:
    for name, df in preprocesados_post_ajustes.items():
        df = df.copy()

        # 1. Cocheras: se imputan nulos en 0.
        if 'cocheras' in df.columns:
            mask = df['cocheras'].isna()
            df['cocheras_imputado'] = mask
            df.loc[mask, 'cocheras'] = 0

            resumen_imputaciones.append({
                'dataset': name,
                'columna': 'cocheras',
                'criterio': 'nulos imputados en 0',
                'n_imputados': int(mask.sum())
            })

        # 2. Moneda de expensas: si hay valor de expensa y falta moneda, se imputa ARS.
        if 'expensas' in df.columns and 'expensas_moneda' in df.columns:
            mask = df['expensas'].notna() & df['expensas_moneda'].isna()
            df['expensas_moneda_imputado'] = mask
            df.loc[mask, 'expensas_moneda'] = 'ARS'

            resumen_imputaciones.append({
                'dataset': name,
                'columna': 'expensas_moneda',
                'criterio': 'si existe expensa y falta moneda, se imputa ARS',
                'n_imputados': int(mask.sum())
            })

        # 3. Valores de referencia desde columnas originales o rangos auxiliares.
        if {'ambientes_ref', 'ambientes'}.issubset(df.columns):
            mask = df['ambientes_ref'].isna() & df['ambientes'].notna()
            df['ambientes_ref_imputado_desde_ambientes'] = mask
            df.loc[mask, 'ambientes_ref'] = df.loc[mask, 'ambientes']
            resumen_imputaciones.append({
                'dataset': name,
                'columna': 'ambientes_ref',
                'criterio': 'ambientes_ref imputado desde ambientes',
                'n_imputados': int(mask.sum())
            })

        df = completar_ref_desde_rango(df, 'ambientes_ref', 'ambientes_min', 'ambientes_max', name)
        df = completar_ref_desde_rango(df, 'dormitorios_ref', 'dormitorios_min', 'dormitorios_max', name)
        df = completar_ref_desde_rango(df, 'banios_ref', 'banios_min', 'banios_max', name)

        # 4. Regla de negocio simple: ambientes = dormitorios + 1.
        if {'ambientes_ref', 'dormitorios_ref'}.issubset(df.columns):
            mask = df['ambientes_ref'].isna() & df['dormitorios_ref'].notna()
            df['ambientes_ref_imputado_desde_dormitorios'] = mask
            df.loc[mask, 'ambientes_ref'] = df.loc[mask, 'dormitorios_ref'] + 1

            resumen_imputaciones.append({
                'dataset': name,
                'columna': 'ambientes_ref',
                'criterio': 'ambientes_ref = dormitorios_ref + 1',
                'n_imputados': int(mask.sum())
            })

        preprocesados_post_ajustes[name] = df

    df_resumen_imputaciones = pd.DataFrame(resumen_imputaciones)
    display(df_resumen_imputaciones[df_resumen_imputaciones['n_imputados'] > 0])

    preprocesados_post_ajustes_con_flags = {}
    for name, df in preprocesados_post_ajustes.items():
        preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
        globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]

else:
    print('No se aplicaron imputaciones controladas.')


Después de la primera imputación, las variables de referencia quedan como columnas principales. Los rangos y columnas originales se revisan solo para verificar si aportan algo adicional antes de eliminarlas como auxiliares.


In [ ]:
# La imputacion de ambientes_ref desde dormitorios_ref se integro en la celda anterior.
# Se deja esta celda como separador para evitar aplicar la misma regla dos veces.
print('Regla ambientes_ref = dormitorios_ref + 1 ya aplicada en imputaciones controladas.')


Aunque `ambientes` y `ambientes_ref` tienen origen distinto, cuando ambas existen suelen coincidir. Por eso se conserva `ambientes_ref` como variable analítica y `ambientes` queda como auxiliar.


In [ ]:
# Comparacion entre ambientes y ambientes_ref, si ambas columnas existen.

comparacion_ambientes = []

for name, df in preprocesados_post_ajustes.items():
    if {'ambientes', 'ambientes_ref'}.issubset(df.columns):
        comparables = df[['ambientes', 'ambientes_ref']].dropna()

        if len(comparables) > 0:
            pct_iguales = (comparables['ambientes'] == comparables['ambientes_ref']).mean() * 100
        else:
            pct_iguales = None

        comparacion_ambientes.append({
            'dataset': name,
            'ambientes_no_nulos': int(df['ambientes'].notna().sum()),
            'ambientes_ref_no_nulos': int(df['ambientes_ref'].notna().sum()),
            'comparables_no_nulos': int(len(comparables)),
            'pct_iguales_entre_ambas': None if pct_iguales is None else round(pct_iguales, 2)
        })

df_comparacion_ambientes = pd.DataFrame(comparacion_ambientes)
display(df_comparacion_ambientes)

In [ ]:
# Eliminacion de columnas auxiliares redundantes frente a variables de referencia.
# Se conservan *_ref como variables analiticas consolidadas.

COLUMNAS_AUXILIARES_REDUNDANTES = {
    'ambientes': 'se conserva ambientes_ref como variable consolidada para analisis',
    'ambientes_min': 'se conserva ambientes_ref; el rango ya fue usado como insumo si correspondia',
    'ambientes_max': 'se conserva ambientes_ref; el rango ya fue usado como insumo si correspondia',
    'dormitorios_min': 'se conserva dormitorios_ref; el rango ya fue usado como insumo si correspondia',
    'dormitorios_max': 'se conserva dormitorios_ref; el rango ya fue usado como insumo si correspondia',
    'banios_min': 'se conserva banios_ref; el rango ya fue usado como insumo si correspondia',
    'banios_max': 'se conserva banios_ref; el rango ya fue usado como insumo si correspondia',
}

resumen_columnas_redundantes = []

for name, df in preprocesados_post_ajustes.items():
    columnas_a_borrar = []

    for col, motivo in COLUMNAS_AUXILIARES_REDUNDANTES.items():
        if col not in df.columns:
            continue

        if col.startswith('ambientes') and 'ambientes_ref' not in df.columns:
            continue
        if col.startswith('dormitorios') and 'dormitorios_ref' not in df.columns:
            continue
        if col.startswith('banios') and 'banios_ref' not in df.columns:
            continue

        columnas_a_borrar.append(col)
        resumen_columnas_redundantes.append({
            'dataset': name,
            'columna_eliminada': col,
            'motivo': motivo
        })

    if columnas_a_borrar:
        preprocesados_post_ajustes[name] = df.drop(columns=columnas_a_borrar)


df_resumen_columnas_redundantes = pd.DataFrame(resumen_columnas_redundantes)

if not df_resumen_columnas_redundantes.empty:
    display(df_resumen_columnas_redundantes)
else:
    print('No se encontraron columnas auxiliares redundantes para eliminar.')

preprocesados_post_ajustes_con_flags = {}
for name, df in preprocesados_post_ajustes.items():
    preprocesados_post_ajustes_con_flags[name] = add_null_flags(df)
    globals()[f'{name}_post_ajustes'] = preprocesados_post_ajustes_con_flags[name]


### 4.5 Variables susceptibles de reconstrucción

Algunas variables no conviene imputarlas directamente, pero sí pueden reconstruirse desde otras columnas o desde texto. Se mantienen para una etapa posterior si son relevantes para el análisis.


In [ ]:
COLUMNAS_SUSCEPTIBLES_RECONSTRUCCION = {
    'expensas': 'buscar en expensas_texto, atributos_texto o descripcion',
    'cocheras': 'buscar menciones en atributos_texto, titulo o descripcion',
    'dormitorios_ref': 'revisar dormitorios_min/max, ambientes_ref y texto',
    'estado_entrega': 'buscar menciones de pozo, estrenar, usado, entrega, posesion',
    'posesion': 'buscar fechas o condiciones de posesion en texto',
    'superficie_ref_m2': 'recalcular solo si hay superficie cubierta/total comparable',
    'precio_m2_ref': 'recalcular solo si existen precio y superficie_ref_m2 validos',
}

for col, criterio in COLUMNAS_SUSCEPTIBLES_RECONSTRUCCION.items():
    print(f'{col}: {criterio}')


## 5. Guardado final de datasets post-ajustes

El guardado se hace al final del notebook para que los archivos incluyan todas las decisiones tomadas en esta etapa: ajustes puntuales, separación por operación, eliminación de columnas y reglas de imputación/reconstrucción que se hayan activado.


In [ ]:
# Guardado final post-ajustes.
# Se guardan todos los datasets de trabajo, no solo los cuatro usados para comparar matrices.

for name, df in preprocesados_post_ajustes_con_flags.items():
    path = PROCESSED_DIR / f'{name}_post_ajustes_con_flags_nulos_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado final post-ajustes:', path.relative_to(REPO_ROOT), df.shape)
